# 02 — Data Preprocessing & Leakage-Free Pipeline

**CustomerIQ — Customer Intelligence & Churn Prediction Platform**

---

## Objectives

In this notebook, we transform our raw data into model-ready numerical matrices while strictly adhering to **production data leakage prevention rules**:

1. **Cleaning Anomalies**: Resolve the whitespace values in `TotalCharges` and drop identifiers (`customerID`).
2. **Stratified Train/Test Split**: Split data BEFORE fitting any scaler or encoder.
3. **Feature Scaling**: Standardize numerical features (`tenure`, `MonthlyCharges`, `TotalCharges`).
4. **Categorical Encoding**: One-Hot Encode categorical features with `handle_unknown='ignore'`.
5. **Scikit-Learn ColumnTransformer**: Build a unified, reusable pipeline.
6. **Verification & Proof of Zero Leakage**: Demonstrate mathematically that the test set was not exposed during fitting.
7. **Serialization**: Save the processed splits and the fitted preprocessor for modeling and API serving.

## 1. Setup & Environment

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from config import (
    RAW_DATASET_PATH, DATA_PROCESSED, MODELS_DIR,
    TARGET_COLUMN, TEST_SIZE, RANDOM_STATE
)
from src.data.make_dataset import load_raw_data, clean_dataset, split_data
from src.features.build_features import (
    get_feature_lists, build_preprocessor, get_transformed_feature_names
)

print("Modules imported successfully.")

Modules imported successfully.


---
## 2. Load & Clean Raw Data

Recall our discovery in Phase 1:
- `TotalCharges` has 11 rows with whitespace string `' '`.
- All 11 rows have `tenure = 0` (customers who just joined and have not yet completed a billing cycle).
- Solution: Impute `TotalCharges` with `0.0` for these rows, convert the column to `float64`, drop `customerID`, and map `Churn` (`Yes` -> 1, `No` -> 0).

In [2]:
# Load raw dataset
df_raw = load_raw_data(RAW_DATASET_PATH)
print(f"Raw dataset shape: {df_raw.shape}")

# Clean anomalies
df_clean = clean_dataset(df_raw)
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"TotalCharges dtype: {df_clean['TotalCharges'].dtype}")
print(f"Any NaN values left?: {df_clean.isna().sum().sum()}")
print(f"Target distribution:\n{df_clean[TARGET_COLUMN].value_counts(normalize=True).round(3)}")

Raw dataset shape: (7043, 21)
Cleaned dataset shape: (7043, 20)
TotalCharges dtype: float64
Any NaN values left?: 0
Target distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


---
## 3. The Golden Rule: Split BEFORE Preprocessing (Preventing Data Leakage)

### What is Data Leakage?
Data leakage occurs when information from outside the training dataset is used to create or fit the model.

**The Classical Mistake:**
```python
# ❌ INCORRECT (Leaky):
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Calculated mean and std across ALL data, including test!
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```
Why is that bad? Because `scaler.fit(X)` computes $\mu$ and $\sigma$ using the test records. In production, future customers do not exist yet when training! If your test set influences the scaler, your test performance will be artificially optimistic and will not generalize.

**The Correct Approach:**
```python
# ✅ CORRECT (Leakage-Free):
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Learn mu and sigma ONLY from train
X_test_scaled = scaler.transform(X_test)        # Apply train mu and sigma to test
```

### Why Stratified Sampling?
Our dataset has a 26.5% churn rate. A completely random split might accidentally put 30% churners in Train and 15% in Test. `stratify=y` ensures that both the train and test subsets match the exact population churn ratio.

In [3]:
# Execute the stratified split
X_train, X_test, y_train, y_test = split_data(
    df_clean,
    target_column=TARGET_COLUMN,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape} | y_test shape:  {y_test.shape}")
print()
print(f"Train churn proportion: {y_train.mean():.4f} ({y_train.sum()} churners)")
print(f"Test churn proportion:  {y_test.mean():.4f} ({y_test.sum()} churners)")

X_train shape: (5634, 19) | y_train shape: (5634,)
X_test shape:  (1409, 19) | y_test shape:  (1409,)

Train churn proportion: 0.2654 (1495 churners)
Test churn proportion:  0.2654 (374 churners)


---
## 4. Feature Transformations: Scaling & Encoding

### A. Numerical Scaling (StandardScaler)
- Features like `tenure` (0–72) and `TotalCharges` (0–8684) have vastly different scales.
- Distance-based algorithms (KNN, SVM) and gradient-based algorithms (Logistic Regression) will be dominated by large-scale features unless normalized.
- Standard score: $z = \frac{x - \mu}{\sigma}$

### B. Categorical Encoding (OneHotEncoder)
- Multi-category features like `PaymentMethod` (4 categories) or `InternetService` (3 categories) cannot be passed as strings to ML algorithms.
- Why not Label Encoding (1, 2, 3)? Because label encoding introduces a false numerical ordering (e.g. "Electronic check" = 3 > "Mailed check" = 1).
- We use **One-Hot Encoding** with `handle_unknown='ignore'`, which converts categories into binary indicator columns ($0$ or $1$) and handles any future unseen categories gracefully in production.

In [4]:
# Separate numerical and categorical feature lists
num_cols, cat_cols = get_feature_lists(X_train)

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print()
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

Numerical features (3): ['MonthlyCharges', 'TotalCharges', 'tenure']

Categorical features (16): ['Contract', 'Dependents', 'DeviceProtection', 'InternetService', 'MultipleLines', 'OnlineBackup', 'OnlineSecurity', 'PaperlessBilling', 'Partner', 'PaymentMethod', 'PhoneService', 'SeniorCitizen', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'gender']


In [5]:
# Build the Scikit-Learn ColumnTransformer
preprocessor = build_preprocessor(num_cols, cat_cols)
print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['MonthlyCharges', 'TotalCharges', 'tenure']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Contract', 'Dependents', 'DeviceProtection',
                                  'InternetService', 'MultipleLines',
                                  'OnlineBackup', 'OnlineSecurity',
                                  'PaperlessBilling', 'Partner',
                                  'PaymentMethod', 'PhoneService',
                                  'SeniorCitizen', 'StreamingMovies',
                                  'StreamingTV', 'TechSupport', 'gender'])])


---
## 5. Fit Preprocessor on Training Data & Transform

We now execute:
1. `preprocessor.fit(X_train)`: Learns means, variances, and category lists from `X_train` only.
2. `preprocessor.transform(X_train)`: Scales and encodes the training data.
3. `preprocessor.transform(X_test)`: Applies the learned transformations to the test data without ever recalculating statistics.

In [6]:
# Fit on training data ONLY
preprocessor.fit(X_train)

# Transform both splits
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Extract human-readable feature names
feature_names = get_transformed_feature_names(preprocessor, num_cols, cat_cols)

print(f"Transformed X_train shape: {X_train_transformed.shape}")
print(f"Transformed X_test shape:  {X_test_transformed.shape}")
print(f"Total engineered features: {len(feature_names)}")
print()
print("Sample of first 10 feature names:")
for name in feature_names[:10]:
    print(f"  - {name}")

Transformed X_train shape: (5634, 46)
Transformed X_test shape:  (1409, 46)
Total engineered features: 46

Sample of first 10 feature names:
  - num__MonthlyCharges
  - num__TotalCharges
  - num__tenure
  - cat__Contract_Month-to-month
  - cat__Contract_One year
  - cat__Contract_Two year
  - cat__Dependents_No
  - cat__Dependents_Yes
  - cat__DeviceProtection_No
  - cat__DeviceProtection_No internet service


---
## 6. Mathematical Verification of Zero Data Leakage

Let's mathematically verify that we did NOT leak test data into the scaler:
- For `X_train`, the mean of numerical features should be **exactly 0.0** and standard deviation **1.0**.
- For `X_test`, because it was scaled using `X_train`'s statistics, its mean will be **close to 0.0, but NOT exactly 0.0**.

If `X_test` had a mean of exactly 0.0, it would be undeniable proof of data leakage (meaning the scaler was fitted on the test data).

In [7]:
# Inspect the first numerical feature (tenure) in both splits
tenure_train = X_train_transformed[:, 0]
tenure_test = X_test_transformed[:, 0]

print("Tenure in Training Set:")
print(f"  Mean: {tenure_train.mean():.6f} (Expected: ~0.0)")
print(f"  Std:  {tenure_train.std():.6f}  (Expected: ~1.0)")

print("\nTenure in Test Set (Transformed with Train Statistics):")
print(f"  Mean: {tenure_test.mean():.6f} (Expected: close to 0, but not exactly 0)")
print(f"  Std:  {tenure_test.std():.6f}  (Expected: close to 1, but not exactly 1)")

assert np.isclose(tenure_train.mean(), 0.0, atol=1e-5), "Train mean must be zero!"
print("\n✅ Zero Data Leakage Confirmed: Preprocessing is fully compliant.")

Tenure in Training Set:
  Mean: -0.000000 (Expected: ~0.0)
  Std:  1.000000  (Expected: ~1.0)

Tenure in Test Set (Transformed with Train Statistics):
  Mean: -0.027911 (Expected: close to 0, but not exactly 0)
  Std:  0.991779  (Expected: close to 1, but not exactly 1)

✅ Zero Data Leakage Confirmed: Preprocessing is fully compliant.


---
## 7. Persist Processed Data & Fitted Pipeline

We save:
1. `preprocessor.joblib`: The fitted transformer artifact that will be loaded during Phase 3-7 and in our Phase 8 FastAPI serving.
2. `X_train.parquet`, `X_test.parquet`, `y_train.csv`, `y_test.csv`: The processed data partitions.

In [8]:
# Convert transformed arrays into DataFrames with clear column names
df_train_proc = pd.DataFrame(X_train_transformed, columns=feature_names)
df_test_proc = pd.DataFrame(X_test_transformed, columns=feature_names)

# Ensure output directories exist
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save processed feature DataFrames
df_train_proc.to_csv(DATA_PROCESSED / 'X_train_processed.csv', index=False)
df_test_proc.to_csv(DATA_PROCESSED / 'X_test_processed.csv', index=False)
y_train.to_csv(DATA_PROCESSED / 'y_train.csv', index=False)
y_test.to_csv(DATA_PROCESSED / 'y_test.csv', index=False)

# Save raw feature splits (useful for end-to-end Pipeline training and inference testing)
X_train.to_csv(DATA_PROCESSED / 'X_train_raw.csv', index=False)
X_test.to_csv(DATA_PROCESSED / 'X_test_raw.csv', index=False)

# Save fitted preprocessor
preprocessor_path = MODELS_DIR / 'preprocessor.joblib'
joblib.dump(preprocessor, preprocessor_path)

print(f"Processed datasets saved to: {DATA_PROCESSED}")
print(f"Fitted preprocessor pipeline saved to: {preprocessor_path}")
print("\nPhase 2 completed successfully!")

Processed datasets saved to: C:\Users\jasir\Downloads\ML_PROJ\CustomerIQ\data\processed
Fitted preprocessor pipeline saved to: C:\Users\jasir\Downloads\ML_PROJ\CustomerIQ\models\preprocessor.joblib

Phase 2 completed successfully!
